# 01 - Environment Smoke Test

This notebook validates your local **llm-observability-stack** baseline on k3s before deeper tutorials.

Coverage in this notebook:
- Python 3.11 and notebook dependencies
- host tooling (`kubectl`, `helm`, `k3s`, `nvidia-smi`)
- cluster/pod/service health checks
- memory and GPU visibility checks
- active runtime profile from `values.local-k3s.yaml`


In [ ]:
import os
import sys
from pathlib import Path

expected = "/usr/local/bin/python3.11"
print(f"Python executable: {sys.executable}")
print(f"Python version   : {sys.version.split()[0]}")
if Path(sys.executable).resolve().as_posix() != expected:
    print(f"WARNING: This notebook is designed for {expected}.")
    print("Switch the notebook kernel to Python 3.11 before continuing.")
else:
    print("Kernel check passed.")


In [ ]:
import importlib
import subprocess
import sys

packages = ["requests", "pandas", "matplotlib", "pyyaml", "langsmith", "tabulate"]
missing = []
for pkg in packages:
    module_name = "yaml" if pkg == "pyyaml" else pkg
    try:
        importlib.import_module(module_name)
    except Exception:
        missing.append(pkg)

if missing:
    print("Installing missing packages:", ", ".join(missing))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])
else:
    print("All common packages are already installed.")


In [ ]:
import json
import os
import shlex
import subprocess
from pathlib import Path


def run(cmd: str, check: bool = True):
    print(f"$ {cmd}")
    proc = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    if proc.stdout:
        print(proc.stdout.rstrip())
    if proc.stderr:
        print(proc.stderr.rstrip())
    if check and proc.returncode != 0:
        raise RuntimeError(f"Command failed ({proc.returncode}): {cmd}")
    return proc


def run_json(cmd: str):
    proc = run(cmd)
    return json.loads(proc.stdout)


PROJECT_ROOT = Path("/media/waqasm86/External1/Project-Nvidia-Office/Project-Llamatelemetry/langchain-kubernetes-jupyterlab/llm-observability-stack")
NAMESPACE = "llm-observability"
print(f"Project root: {PROJECT_ROOT}")


## Toolchain Checks
The commands below should all resolve on your host.


In [ ]:
for cmd in [
    "which kubectl",
    "kubectl version --client",
    "which helm",
    "helm version --short",
    "which k3s",
    "k3s --version",
    "which nvidia-smi",
]:
    run(cmd, check=False)


## Kubernetes Inventory


In [ ]:
run("kubectl get nodes -o wide")
run(f"kubectl get pods -n {NAMESPACE} -o wide")
run(f"kubectl get svc -n {NAMESPACE}")


## Cluster Resource Snapshot


In [ ]:
run("kubectl top nodes", check=False)
run(f"kubectl top pods -n {NAMESPACE}", check=False)


## GPU Snapshot


In [ ]:
run("nvidia-smi", check=False)


## Runtime Profile From Values


In [ ]:
import yaml

values_path = PROJECT_ROOT / "values.local-k3s.yaml"
with values_path.open() as f:
    values = yaml.safe_load(f)

profile = {
    "gpu_resource": values.get("nvidia", {}).get("gpuResourceName"),
    "gpu_count": values.get("nvidia", {}).get("gpuCount"),
    "langsmith_project": values.get("langsmith", {}).get("project"),
    "ollama_service_type": values.get("ollama", {}).get("service", {}).get("type"),
    "langchain_service_type": values.get("langchainDemo", {}).get("service", {}).get("type"),
    "open_webui_service_type": values.get("openWebUI", {}).get("service", {}).get("type"),
    "python_toolbox_enabled": values.get("pythonToolbox", {}).get("enabled"),
    "seeder_enabled": values.get("langsmithDashboardSeeder", {}).get("enabled"),
    "redis_enabled": values.get("redis", {}).get("enabled"),
}

profile


## Optional Local Endpoint Checks
Open WebUI is exposed on localhost in your current profile.


In [ ]:
import requests

for url in ["http://localhost:8080/", "http://localhost:8080/health", "http://localhost:8080/api/version"]:
    try:
        r = requests.get(url, timeout=5)
        print(url, "->", r.status_code)
    except Exception as exc:
        print(url, "->", exc)


## Done
Proceed to `02-ollama-api-basics.ipynb` after this notebook shows healthy outputs.
